In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('..')
from src import functions as fc

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier

In [3]:
df_modeling = fc.load_data_clean("bank_final.csv")

Buscando archivo en: /Users/jbp/Desktop/IRONHACK/SEMANA7/ML_project/data/cleaned/bank_final.csv


In [4]:
features = df_modeling.drop(columns=["target", "duration"])
target = df_modeling["target"]

In [5]:
x_train, x_test, y_train, y_test = train_test_split(features, target, test_size=0.20, random_state=0)

In [ ]:
# 1. Definimos el escalador
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)

x_test_scaled = scaler.transform(x_test)

Gradient Boosting:

In [11]:
gb_class = GradientBoostingClassifier(max_depth=3, n_estimators=100, learning_rate=0.1, random_state=42)

In [12]:
gb_class.fit(x_train_scaled, y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [13]:
pred = gb_class.predict(x_test_scaled)

In [14]:
accuracy = gb_class.score(x_test_scaled, y_test)
print(f"La precisión del modelo es: {accuracy:.2f}")

La precisión del modelo es: 0.64


In [15]:
importancias_gb = pd.DataFrame({
    'feature': features.columns, 
    'importance': gb_class.feature_importances_
}).sort_values('importance', ascending=False)

print("Top variables para Gradient Boosting:")
print(importancias_gb.head(5))

Top variables para Gradient Boosting:
          feature  importance
7       euribor3m    0.436193
5  cons.price.idx    0.101183
0             age    0.081741
1        campaign    0.050199
6   cons.conf.idx    0.033758


Random search:

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, loguniform

param_dist_gb = {
    'n_estimators': randint(200, 800),
    'learning_rate': loguniform(0.001, 0.3),
    'max_depth': randint(2, 6),
    'subsample': [0.6, 0.8, 1.0],
    'min_samples_split': randint(2, 30),
    'min_samples_leaf': randint(1, 15),
    'max_features': ['sqrt', 'log2', None]}

random_gb = RandomizedSearchCV(
    estimator=GradientBoostingClassifier(),
    param_distributions=param_dist_gb,
    n_iter=70,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=2,
    random_state=42)

random_gb.fit(x_train_scaled, y_train)

Fitting 5 folds for each of 70 candidates, totalling 350 fits
[CV] END learning_rate=0.008468008575248327, max_depth=2, max_features=None, min_samples_leaf=11, min_samples_split=9, n_estimators=220, subsample=1.0; total time=   0.9s
[CV] END learning_rate=0.008468008575248327, max_depth=2, max_features=None, min_samples_leaf=11, min_samples_split=9, n_estimators=220, subsample=1.0; total time=   1.0s
[CV] END learning_rate=0.008468008575248327, max_depth=2, max_features=None, min_samples_leaf=11, min_samples_split=9, n_estimators=220, subsample=1.0; total time=   1.0s
[CV] END learning_rate=0.008468008575248327, max_depth=2, max_features=None, min_samples_leaf=11, min_samples_split=9, n_estimators=220, subsample=1.0; total time=   1.0s
[CV] END learning_rate=0.008468008575248327, max_depth=2, max_features=None, min_samples_leaf=11, min_samples_split=9, n_estimators=220, subsample=1.0; total time=   1.1s
[CV] END learning_rate=0.001124579825911934, max_depth=3, max_features=log2, min_sa

,estimator,GradientBoostingClassifier()
,param_distributions,"{'learning_rate': <scipy.stats....t 0x1797eb620>, 'max_depth': <scipy.stats....t 0x17994f250>, 'max_features': ['sqrt', 'log2', ...], 'min_samples_leaf': <scipy.stats....t 0x1797e7100>, ...}"
,n_iter,70
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [ ]:
print("Mejores parámetros:", random_gb.best_params_)

best_model_bag = random_gb.best_estimator_

pred_bag = best_model_bag.predict(x_test_scaled)

accuracy_bag = best_model_bag.score(x_test_scaled, y_test)
print(f"La precisión del modelo es: {accuracy_bag:.2f}")

Mejores parámetros: {'learning_rate': np.float64(0.1974451054660558), 'max_depth': 4, 'max_features': None, 'min_samples_leaf': 8, 'min_samples_split': 15, 'n_estimators': 451, 'subsample': 0.6}
La precisión del modelo es: 0.58


In [20]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision_bag = precision_score(y_test, pred_bag)
recall_bag = recall_score(y_test, pred_bag)
f1_bag = f1_score(y_test, pred_bag)

print(f"Precision del modelo es: {precision_bag:.2f}")
print(f"Recall del modelo es: {recall_bag:.2f}")
print(f"F1-score del modelo es: {f1_bag:.2f}")

Precision del modelo es: 0.48
Recall del modelo es: 0.46
F1-score del modelo es: 0.47
